# BiasiswaBot · Topic 3 data pipeline

PDF/TXT pathway (TXT input). Two section notes from the official PTPTN education-loan page feed the Topic 2 Knowledge Source for FR1, FR2 and FR8.

Run all cells in order. In Colab, open this notebook from the semester repository and leave the runtime at its default working directory. Cell 1 clones the repository into the temporary workspace if needed. Locally, start Jupyter from the repository root or its notebooks directory. The notebook uses only the Python standard library. Cell 2 is optional for the course dummy ZIP and is skipped here.

Inputs are original, manually written notes based on the Malay source. The notebook's executable scope begins with these pinned TXT files. Use the official page for exact wording and current conditions.

In [1]:
# Cell 1 — Set up the workspace
from pathlib import Path
import subprocess

COLAB = Path("/content").is_dir()
if COLAB:
    BASE_DIR = Path("/content/week3_project")
    if not (BASE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/safiahshfrn/ai-engineering-biasiswa-assistant.git",
            str(BASE_DIR),
        ], check=True)
else:
    BASE_DIR = next(
        (folder for folder in (Path.cwd(), *Path.cwd().parents)
         if (folder / "docs/week2_requirements.md").is_file()),
        None,
    )
    if BASE_DIR is None:
        raise FileNotFoundError("Start Jupyter from the BiasiswaBot repository.")

RAW_DIR = RAW = BASE_DIR / "data/raw"
PROCESSED_DIR = PROCESSED = OUT = BASE_DIR / "data/processed"
BASE = BASE_DIR
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Raw: data/raw | Part B export: data/processed")


Raw: data/raw | Part B export: data/processed


In [2]:
# Cell 3 — Verify the exact source inputs
SOURCE_FILES = {
    "S01": ("week3/ptptn_postgraduate_coverage.txt", "Pinjaman Pendidikan Yang Ditawarkan"),
    "S02": ("week3/ptptn_application_conditions.txt", "Syarat-Syarat Permohonan"),
}
for source_id, (relative_path, heading) in SOURCE_FILES.items():
    path = RAW_DIR / relative_path
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty source: {relative_path}. Restore the tracked TXT file.")
    print(f"{source_id} | {relative_path} | {path.stat().st_size} bytes")


S01 | week3/ptptn_postgraduate_coverage.txt | 356 bytes
S02 | week3/ptptn_application_conditions.txt | 356 bytes


In [3]:
# Cell 4 — Explicit source metadata, ingestion and inspection
import hashlib
import json
import re
import textwrap
from datetime import datetime, timezone
from tempfile import TemporaryDirectory

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def relative(path):
    return path.relative_to(BASE_DIR).as_posix()

def inspect_sources():
    sources = []
    for source_id, (name, heading) in SOURCE_FILES.items():
        path = RAW_DIR / name
        raw_text = path.read_text(encoding="utf-8")
        paragraphs = [
            " ".join(block.split())
            for block in re.split(r"\n[ \t]*\n", raw_text.strip())
            if block.strip()
        ]
        if not paragraphs:
            raise ValueError(f"No readable paragraphs in {name}. Restore the curated UTF-8 source.")
        metadata = {
            "source_id": source_id,
            "source_filename": relative(path),
            "source_sha256": digest(path),
            "source_bytes": path.stat().st_size,
            "source_title": "Pinjaman Pendidikan",
            "publisher": "Perbadanan Tabung Pendidikan Tinggi Nasional (PTPTN)",
            "source_url": "https://www.ptptn.gov.my/pinjaman-pendidikan/",
            "source_section": heading,
            "source_date": None,
            "reviewed_on": "2026-11-16",
            "source_version": "Curated English section notes v1",
            "access_rights": "Public reference page. Repository contains original paraphrases only.",
            "provenance_limit": "Fingerprint covers the local TXT notes. Publication date and HTML version are unverified.",
        }
        sources.append((metadata, paragraphs))
    return sources

sources = inspect_sources()
for metadata, paragraphs in sources:
    print(metadata["source_id"], metadata["source_section"], "|", len(paragraphs), "paragraphs")
    print(paragraphs[0])


S01 Pinjaman Pendidikan Yang Ditawarkan | 2 paragraphs
The PTPTN education-loan table lists public higher education institutions (IPTA) for both master's and doctoral study. The entries cover full-time, part-time and distance learning.
S02 Syarat-Syarat Permohonan | 2 paragraphs
PTPTN requires an offer of admission that meets the applicable entry standards. The course must have Ministry of Higher Education approval and be registered with PTPTN.


In [4]:
# Cell 4A — Initialize the common lineage registry
lineage_entries = []
CONFIG = {"strategy": "paragraph", "encoding": "utf-8", "whitespace": "collapse within paragraphs"}


In [5]:
# Cell 5A — Compare approaches on identical TXT inputs
def chunks_for(paragraphs, strategy):
    if strategy == "paragraph":
        return paragraphs
    if strategy == "fixed_150":
        return textwrap.wrap(" ".join(paragraphs), width=150,
                             break_long_words=False, break_on_hyphens=False)
    raise ValueError(f"Unknown chunking strategy: {strategy}")

comparison = {}
for strategy in ("fixed_150", "paragraph"):
    comparison[strategy] = {}
    for metadata, paragraphs in sources:
        chunks = chunks_for(paragraphs, strategy)
        comparison[strategy][metadata["source_id"]] = {
            "chunks": len(chunks),
            "max_chars": max(map(len, chunks)),
            "intact_paragraphs": sum(any(p in c for c in chunks) for p in paragraphs),
            "source_paragraphs": len(paragraphs),
        }
print(json.dumps(comparison, indent=2))

def prepare(sources, strategy):
    return [
        {
            "chunk_id": f'{metadata["source_id"]}:p{index}',
            "paragraph": index,
            "text": chunk,
            **metadata,
        }
        for metadata, paragraphs in sources
        for index, chunk in enumerate(chunks_for(paragraphs, strategy), 1)
    ]

def store_validated(output, records, inputs):
    output.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=output.parent) as staging:
        candidate = Path(staging) / output.name
        candidate.write_text(
            "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in records),
            encoding=CONFIG["encoding"],
        )
        validate(candidate, records, inputs)
        candidate.replace(output)
    return validate(output, records, inputs)


{
  "fixed_150": {
    "S01": {
      "chunks": 3,
      "max_chars": 147,
      "intact_paragraphs": 0,
      "source_paragraphs": 2
    },
    "S02": {
      "chunks": 3,
      "max_chars": 149,
      "intact_paragraphs": 0,
      "source_paragraphs": 2
    }
  },
  "paragraph": {
    "S01": {
      "chunks": 2,
      "max_chars": 180,
      "intact_paragraphs": 2,
      "source_paragraphs": 2
    },
    "S02": {
      "chunks": 2,
      "max_chars": 185,
      "intact_paragraphs": 2,
      "source_paragraphs": 2
    }
  }
}


In [6]:
# B3 — Validate and review the selected preparation output
def validate(path, expected, inputs):
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError(f"Missing or empty export: {path.name}")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    expected_paragraphs = {
        (metadata["source_id"], index): (metadata, paragraph)
        for metadata, paragraphs in inputs
        for index, paragraph in enumerate(paragraphs, 1)
    }
    observed = {(record["source_id"], record["paragraph"]) for record in records}
    if (records != expected or not records or observed != set(expected_paragraphs)
            or len({record["chunk_id"] for record in records}) != len(records)
            or len(observed) != len(records)):
        raise ValueError("Export has missing, changed or duplicate records.")
    for record in records:
        metadata, paragraph = expected_paragraphs[record["source_id"], record["paragraph"]]
        if (not paragraph or record["text"] != paragraph
                or any(record[key] != value for key, value in metadata.items())):
            raise ValueError(f'Invalid text or lineage: {record["chunk_id"]}')
    return {"status": "passed", "records": len(records),
            "bytes": path.stat().st_size, "sha256": digest(path)}

part_b_path = PROCESSED_DIR / "prepared_text.jsonl"
part_b_records = prepare(sources, CONFIG["strategy"])
part_b_validation = store_validated(part_b_path, part_b_records, sources)
print(relative(part_b_path), part_b_validation)
sample = part_b_records[2]
print(sample["chunk_id"], sample["source_section"], sample["text"])


data/processed/prepared_text.jsonl {'status': 'passed', 'records': 4, 'bytes': 3810, 'sha256': 'ca88a2ae4cacb69edc5657d53e04ea580a278ad33d398c6254a8c543f4698baa'}
S02:p1 Syarat-Syarat Permohonan PTPTN requires an offer of admission that meets the applicable entry standards. The course must have Ministry of Higher Education approval and be registered with PTPTN.


In [7]:
# B4.1 / Cell 6A — Register and export common lineage
def map_output(inputs, records, output):
    return [
        {
            **metadata,
            "prepared_output": relative(output),
            "prepared_sha256": digest(output),
            "chunk_ids": [r["chunk_id"] for r in records if r["source_id"] == metadata["source_id"]],
        }
        for metadata, paragraphs in inputs
    ]

def write_json(path, value):
    with TemporaryDirectory(dir=path.parent) as staging:
        target = Path(staging) / path.name
        target.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        target.replace(path)

lineage_entries.extend(map_output(sources, part_b_records, part_b_path))
part_b_manifest = BASE_DIR / "data/week3_manifest.json"
write_json(part_b_manifest, {
    "schema_version": 1,
    "pathway": "documents",
    "processing": CONFIG,
    "comparison": comparison,
    "validation": part_b_validation,
    "lineage": lineage_entries,
})
print("Saved", relative(part_b_manifest))


Saved data/week3_manifest.json


In [8]:
# Cell 6B — Verify lineage against the saved files
def verify_lineage(entries, records):
    if not entries:
        raise ValueError("No source-to-output mappings.")
    for entry in entries:
        if digest(BASE_DIR / entry["source_filename"]) != entry["source_sha256"]:
            raise ValueError(f'Source fingerprint mismatch: {entry["source_id"]}')
        if digest(BASE_DIR / entry["prepared_output"]) != entry["prepared_sha256"]:
            raise ValueError(f'Output fingerprint mismatch: {entry["prepared_output"]}')
        expected_ids = [record["chunk_id"] for record in records
                        if record["source_id"] == entry["source_id"]]
        if not expected_ids or expected_ids != entry["chunk_ids"]:
            raise ValueError(f'Missing or inconsistent chunk mapping: {entry["source_id"]}')

loaded = json.loads(part_b_manifest.read_text(encoding="utf-8"))
assert loaded["lineage"] == lineage_entries
verify_lineage(loaded["lineage"], part_b_records)
print(f"PASS: verified {len(loaded['lineage'])} source-to-output mappings")


PASS: verified 2 source-to-output mappings


In [9]:
# Cell 7 — Define the reusable V1 batch pipeline
def run_documents(output_dir):
    inputs = inspect_sources()
    records = prepare(inputs, CONFIG["strategy"])
    output = output_dir / "prepared_text.jsonl"
    validation = store_validated(output, records, inputs)
    mapping = map_output(inputs, records, output)
    verify_lineage(mapping, records)
    manifest = {
        "schema_version": 1,
        "pathway": "documents",
        "status": "SUCCESS",
        "execution_time_utc": datetime.now(timezone.utc).isoformat(),
        "processing": CONFIG,
        "sources": [metadata for metadata, _ in inputs],
        "inspection": [{"source_id": metadata["source_id"], "paragraphs": len(paragraphs)}
                       for metadata, paragraphs in inputs],
        "output_count": 1,
        "validation": validation,
        "source_output_mapping": mapping,
    }
    write_json(output_dir / "pipeline_manifest.json", manifest)
    return manifest

print("Cell 7 ready")


Cell 7 ready


In [10]:
# Cell 8A — Execute the documents pipeline
pipeline_dir = BASE_DIR / "data/pipeline_v1/documents"
first = run_documents(pipeline_dir)
assert first["validation"] == part_b_validation, "Part B and Part C outputs differ."
print("SUCCESS:", len(first["sources"]), "TXT inputs →", first["validation"]["records"], "records")
print("Output matches Part B byte for byte.")
print(relative(pipeline_dir / "pipeline_manifest.json"))


SUCCESS: 2 TXT inputs → 4 records
Output matches Part B byte for byte.
data/pipeline_v1/documents/pipeline_manifest.json


In [11]:
# Cell 9 — Rerun independently and compare provenance and outputs
second = run_documents(pipeline_dir)
for field in ("sources", "inspection", "processing", "output_count", "validation", "source_output_mapping"):
    assert first[field] == second[field], f"Repeatability mismatch: {field}"

assert json.loads((pipeline_dir / "pipeline_manifest.json").read_text(encoding="utf-8")) == second
print("PASS: documents — source and output SHA-256, sizes, counts and mappings match.")
print("Output SHA-256:", second["validation"]["sha256"])


PASS: documents — source and output SHA-256, sizes, counts and mappings match.
Output SHA-256: ca88a2ae4cacb69edc5657d53e04ea580a278ad33d398c6254a8c543f4698baa
